# Suspicion Reason Risk Scoring (Synthetic + MNLI)

This notebook builds a synthetic dataset for `reason_for_suspicion`, splits it into train/test, exports an Excel file, and shows a MNLI-based zero-shot approach to score risk.

Note: The MNLI model download requires internet and can take a while the first time.


In [19]:
!pip -q install openpyxl transformers torch
# If you need CPU-only torch on Windows, visit https://pytorch.org/get-started/locally/ for the right install line. 


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)


In [ ]:
# Synthetic dataset generation

def sample_amount(low=5000, high=500000):
    return f'${int(rng.integers(low, high)):,}'

def sample_country():
    return rng.choice(['Country X', 'Country Y', 'Country Z'])

def sample_channel():
    return rng.choice(['cash deposit', 'wire', 'crypto transfer', 'check', 'ACH'])

TEMPLATES = {
    'money_laundering': [
        'Multiple {channel} transactions just under reporting threshold totaling {amt}.',
        'Layering activity through shell companies and offshore accounts; {amt} moved in 48 hours.',
        'Customer insists on third-party payments with no clear business purpose; {amt} involved.',
        'Frequent cash deposits in different branches, structured to avoid detection.',
        'Unusual velocity: rapid in-and-out transfers across unrelated entities.',
        'Source of funds inconsistent with stated income; large inflows of {amt}.',
        'Use of high-risk jurisdictions with complex ownership structures.',
        'Transactions routed through multiple intermediaries with no commercial rationale.',
        'Repeated transfers between related accounts to obscure origin of funds.',
        'Use of money mules to move funds via small transfers adding to {amt}.'
    ],
    'sanctions_evasion': [
        'Payments linked to sanctioned region via intermediaries; {amt} exposure.',
        'Counterparty shares address with known sanctioned entity.',
        'Trade documents appear altered to conceal true destination.',
        'Multiple payments referencing alternate spellings of a blocked party.',
        'Use of third-country routing to mask ultimate beneficiary in {country}.',
        'Customer avoids providing end-user certificates for controlled goods.',
        'Shipping routes inconsistent with declared goods and destination.',
        'Screening hits for sanctioned vessel and related parties.',
        'Funds sent to newly created entity with ties to sanctions list.',
        'Beneficial owner previously flagged for sanctions risk.'
    ],
    'terrorist_financing': [
        'Small, frequent donations to entities with unclear charitable activities.',
        'Transfers to conflict zones without legitimate explanation.',
        'Account activity linked to extremist forums and fundraising campaigns.',
        'Use of prepaid cards and cash withdrawals aligned with known patterns.',
        'Rapid movement of funds to multiple individuals in high-risk areas.',
        'Payments referencing coded language commonly used in illicit fundraising.',
        'Customer refuses to explain relationship to recipients in {country}.',
        'Unusual crypto transfers to wallets flagged by external intelligence.',
        'Funds sent to newly formed NGO with no operating history.',
        'Multiple small transfers to avoid detection of {amt} aggregate.'
    ],
    'fraud': [
        'Chargebacks and disputes spike after new device login.',
        'Customer reports account takeover and unauthorized transfers.',
        'Merchant shows abnormal refund rates and mismatched shipping.',
        'Identity documents appear altered; selfie mismatch flagged.',
        'Multiple accounts opened with similar device fingerprints.',
        'Unusual login pattern from multiple geolocations in one day.',
        'Mismatched billing and shipping addresses across transactions.',
        'Invoices appear duplicated with slight modifications.',
        'Customer attempts to bypass security controls repeatedly.',
        'Card-not-present transactions spike after sudden address change.'
    ],
    'corruption': [
        'Payment request linked to public official with no contract evidence.',
        'Unexplained consulting fees routed to politically exposed person.',
        'Large gifts and hospitality expenses without approvals.',
        'Third-party intermediary requests off-book payment of {amt}.',
        'Vendor selected despite higher cost and weak due diligence.',
        'Charity donations tied to procurement decision makers.',
        'Use of offshore entities to pay for government-related deal.',
        'Unusual commission rates for a high-risk country deal.',
        'Request to split invoice to avoid internal review.',
        'Payments routed through family members of officials.'
    ],
    'kyc_issue': [
        'Customer refuses to provide updated KYC documentation.',
        'Business purpose unclear; activities do not match stated profile.',
        'Ownership structure incomplete after repeated requests.',
        'Significant change in activity without updated source of funds.',
        'Address and phone records mismatch across filings.',
        'Multiple accounts opened with minimal documentation.',
        'Customer uses PO boxes only, no verifiable premises.',
        'Beneficial owners not disclosed despite legal requirement.',
        'Inconsistent employment details across forms.',
        'Customer resists enhanced due diligence questions.'
    ],
    'low_risk': [
        'Customer requested account statement reissue.',
        'Routine payroll deposits consistent with stated income.',
        'Address update due to relocation; provided valid proof.',
        'Single small wire consistent with historical activity.',
        'Customer inquiry about fee schedule; no transaction concerns.',
        'Seasonal increase in sales consistent with business cycle.',
        'Regular vendor payments matching invoices.',
        'Card replacement requested after expiry.',
        'Standard ACH settlement for utility payments.',
        'Balance transfer between own accounts.'
    ]
}

RISK_LABEL = {
    'money_laundering': 'high',
    'sanctions_evasion': 'high',
    'terrorist_financing': 'high',
    'fraud': 'medium',
    'corruption': 'medium',
    'kyc_issue': 'medium',
    'low_risk': 'low'
}

rows = []
for category, templates in TEMPLATES.items():
    for _ in range(20):
        template = rng.choice(templates)
        text = template.format(
            amt=sample_amount(),
            country=sample_country(),
            channel=sample_channel()
        )
        label = RISK_LABEL[category]
        base = {'low': 20, 'medium': 55, 'high': 85}[label]
        urgency = int(np.clip(rng.normal(base, 7), 0, 100))
        rows.append({
            'reason_for_suspicion': text,
            'risk_label': label,
            'category': category,
            'urgency_score': urgency
        })

df = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)
print(df.head(5))
print('Total rows:', len(df))


                                reason_for_suspicion risk_label  \
0  Beneficial owners not disclosed despite legal ...     medium   
1  Multiple accounts opened with similar device f...     medium   
2  Shipping routes inconsistent with declared goo...       high   
3  Beneficial owners not disclosed despite legal ...     medium   
4  Customer refuses to explain relationship to re...       high   

              category  urgency_score  
0            kyc_issue             61  
1                fraud             64  
2    sanctions_evasion             74  
3            kyc_issue             57  
4  terrorist_financing             80  
Total rows: 140


In [22]:
# Train/test split

def split_df(dataframe, test_size=0.2, seed=42):
    try:
        from sklearn.model_selection import train_test_split
        train_df, test_df = train_test_split(
            dataframe,
            test_size=test_size,
            random_state=seed,
            stratify=dataframe['risk_label']
        )
        return train_df.reset_index(drop=True), test_df.reset_index(drop=True)
    except Exception as e:
        print('Falling back to manual split:', e)
        shuffled = dataframe.sample(frac=1, random_state=seed).reset_index(drop=True)
        n_test = int(len(shuffled) * test_size)
        test_df = shuffled.iloc[:n_test].reset_index(drop=True)
        train_df = shuffled.iloc[n_test:].reset_index(drop=True)
        return train_df, test_df

train_df, test_df = split_df(df, test_size=0.2, seed=42)
print('Train:', len(train_df), 'Test:', len(test_df))


Train: 112 Test: 28


In [23]:
# Export to Excel (train/test sheets)

out_path = 'suspicion_train_test.xlsx'
try:
    with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
        train_df.to_excel(writer, sheet_name='train', index=False)
        test_df.to_excel(writer, sheet_name='test', index=False)
    print('Wrote:', out_path)
except Exception as e:
    print('Excel write failed (missing openpyxl?).')
    print('Error:', e)
    # Fallback to CSV to avoid data loss
    train_df.to_csv('suspicion_train.csv', index=False)
    test_df.to_csv('suspicion_test.csv', index=False)
    print('Wrote fallback CSVs: suspicion_train.csv, suspicion_test.csv')


Wrote: suspicion_train_test.xlsx


## MNLI-based risk scoring (zero-shot)

We use a MNLI model to score the `reason_for_suspicion` text against risk labels. This is useful when you have no labeled data.


In [24]:
# MNLI / NLI zero-shot scoring 

RISK_LABELS = [ 
    'money laundering', 
    'sanctions evasion', 
    'terrorist financing', 
    'fraud', 
    'corruption', 
    'benign/low risk' 
] 
 
label_to_risk = { 
    'money laundering': 'high', 
    'sanctions evasion': 'high', 
    'terrorist financing': 'high', 
    'fraud': 'medium', 
    'corruption': 'medium', 
    'benign/low risk': 'low' 
} 
 
try: 
    from transformers import pipeline 
    nli = pipeline('zero-shot-classification', model='facebook/bart-large-mnli') 
 
    def score_reason(text): 
        result = nli(text, RISK_LABELS, multi_label=True) 
        top_label = result['labels'][0] 
        top_score = result['scores'][0] 
        risk = label_to_risk[top_label] 
        base = {'low': 20, 'medium': 55, 'high': 85}[risk] 
        urgency = int(np.clip(base + (top_score - 0.5) * 30, 0, 100)) 
        return { 
            'top_label': top_label, 
            'nli_score': float(top_score), 
            'risk_label': risk, 
            'urgency_score': urgency 
        } 
 
    # Quick demo on a few test rows 
    demo = test_df.sample(5, random_state=42).copy() 
    demo_preds = demo['reason_for_suspicion'].apply(score_reason) 
    demo['nli_pred'] = demo_preds.apply(lambda x: x['risk_label']) 
    demo['nli_top_label'] = demo_preds.apply(lambda x: x['top_label']) 
    demo['nli_score'] = demo_preds.apply(lambda x: x['nli_score']) 
    print(demo[['reason_for_suspicion', 'risk_label', 'nli_pred', 'nli_top_label', 'nli_score']].head()) 
 
    # Accuracy on full test split 
    test_preds = test_df['reason_for_suspicion'].apply(score_reason) 
    test_pred_labels = test_preds.apply(lambda x: x['risk_label']) 
    accuracy = (test_pred_labels == test_df['risk_label']).mean() 
    print('Test accuracy (risk_label vs NLI):', round(float(accuracy), 4)) 
except Exception as e: 
    print('Transformers MNLI setup failed. Install requirements:') 
    print('pip install transformers torch') 
    print('Error:', e) 

Loading weights: 100%|██████████| 515/515 [00:00<00:00, 7025.75it/s]


                                 reason_for_suspicion risk_label nli_pred  \
9   Repeated transfers between related accounts to...       high   medium   
25  Transfers to conflict zones without legitimate...       high   medium   
8   Invoices appear duplicated with slight modific...     medium   medium   
21  Customer resists enhanced due diligence questi...     medium     high   
0   Address update due to relocation; provided val...        low      low   

        nli_top_label  nli_score  
9               fraud   0.715584  
25              fraud   0.656690  
8          corruption   0.274408  
21  sanctions evasion   0.165356  
0     benign/low risk   0.027915  
Test accuracy (risk_label vs NLI): 0.6786
